<a href="https://colab.research.google.com/github/volsarino/-/blob/main/%E3%83%AA%E3%82%A2%E3%83%AB%E3%82%BF%E3%82%A4%E3%83%A0%E3%83%8F%E3%83%B3%E3%83%89%E3%83%88%E3%83%A9%E3%83%83%E3%82%AD%E3%83%B3%E3%82%B0%E5%AE%9F%E9%A8%93.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics

In [2]:
import os
import cv2
import numpy as np
import base64
import time
from ultralytics import YOLO
from IPython.display import display, Javascript
from google.colab.output import eval_js

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
best_model_path = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/runs/hand_yolo_model-2/weights/best.pt'
yolo_model = YOLO(best_model_path)

In [7]:
def start_fast_camera_stream():
    js = Javascript('''
      var video;
      var div = null;
      var stream;
      var captureCanvas;
      var imgElement;

      async function initCamera() {
        div = document.createElement('div');
        video = document.createElement('video');
        video.style.display = 'block';

        stream = await navigator.mediaDevices.getUserMedia({
          video: { width: { ideal: 640 }, height: { ideal: 480 }, frameRate: { ideal: 30 } }
        });

        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = video.videoWidth;
        captureCanvas.height = video.videoHeight;

        imgElement = document.createElement('img');
        div.appendChild(imgElement);
        video.style.display = 'none';
      }

      async function captureFrame() {
        if(!captureCanvas) await initCamera();
        var ctx = captureCanvas.getContext('2d');
        ctx.drawImage(video, 0, 0);
        return captureCanvas.toDataURL('image/jpeg', 0.5);
      }

      function showProcessedFrame(imgData) {
        requestAnimationFrame(() => {
          imgElement.src = imgData;
        });
      }

      window.captureFrame = captureFrame;
      window.showProcessedFrame = showProcessedFrame;
    ''')
    display(js)

In [9]:
start_fast_camera_stream()
time.sleep(1)

track_history = {}

# FPS計測用の変数
fps = 0
frame_count = 0
start_time = time.time()

for _ in range(300):
    t0 = time.time()

    frame_data = eval_js('captureFrame()')
    if not frame_data:
        continue

    #画像デコード
    header, encoded = frame_data.split(',', 1)
    data = base64.b64decode(encoded)
    nparr = np.frombuffer(data, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    #左右反転
    frame = cv2.flip(frame, 1)

    #YOLO実行
    results = yolo_model.track(
        frame,
        persist=True,
        conf=0.45,
        imgsz=320,
        verbose=False
    )[0]

    #描画処理
    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        track_ids = results.boxes.id.int().cpu().numpy()
        confidences = results.boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(boxes, track_ids, confidences):
            x1, y1, x2, y2 = map(int, box)
            center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"ID:{track_id} ({conf:.2f})", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            if track_id not in track_history:
                track_history[track_id] = []
            track_history[track_id].append((center_x, center_y))
            if len(track_history[track_id]) > 20:
                track_history[track_id].pop(0)
            pts = np.array(track_history[track_id], dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [pts], isClosed=False, color=(0, 0, 255), thickness=3)

    # 5. FPSのリアルタイム表示
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed > 1.0:
        fps = frame_count / elapsed
        frame_count = 0
        start_time = time.time()

    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    _, buffer = cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 60])
    img_str = 'data:image/jpeg;base64,' + base64.b64encode(buffer).decode('utf-8')
    eval_js(f'showProcessedFrame("{img_str}")')

print("🏁 追従処理が終了しました。")

<IPython.core.display.Javascript object>

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 416ms
Prepared 1 package in 124ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



KeyboardInterrupt: 